# Building Payment-Enabled AI Agents with Oracle AI Database & Spraay x402

**Partner Notebook** | [Spraay Protocol](https://spraay.app) | [Oracle AI Database](https://www.oracle.com/database/ai/)

---

## Overview

AI agents can reason, retrieve context, and generate responses — but they can't **pay for resources** autonomously. As agents move toward production, they need the ability to pay for premium inference, external data, and services without human intervention.

This notebook demonstrates how to combine:
- **Oracle AI Database** — vector storage, semantic search, and agent memory
- **Spraay x402 Gateway** — autonomous USDC micropayments over HTTP (the [x402 protocol](https://www.x402.org))
- **LangChain** — agent orchestration and tool use

The result is an agent that retrieves context from Oracle's vector store, decides when it needs premium AI inference, and **pays for it automatically** via Spraay's x402 gateway — all in a single agentic loop.

### What is x402?

x402 is an open payment protocol built on HTTP status code 402 ("Payment Required"). When an agent requests a paid resource, the server responds with a `402` status and payment requirements. The agent's payment middleware automatically signs a USDC payment on Base and retries the request with a payment proof header. No API keys, no subscriptions — just pay-per-request.

### What is Spraay?

[Spraay Protocol](https://spraay.app) is a multi-chain batch payment protocol deployed across **15 blockchains** (Base, Ethereum, Bitcoin, Arbitrum, Polygon, BNB Chain, Avalanche, Unichain, Plasma, BOB, Bittensor, Solana, Stacks, Stellar, and XRP Ledger). The [Spraay x402 Gateway](https://gateway.spraay.app) exposes **88 pay-per-call endpoints** across **19 categories** — AI inference (200+ models), Bittensor decentralized AI, search/RAG, batch payments, escrow, RPC, IPFS storage, robotics (RTP), supply chain (SCTP), agent wallets (ERC-4337), and more.

Spraay's batch payment tool is an official community tool in [Google's Agent Development Kit (ADK)](https://github.com/google/adk-python-community) — merged as PR #95.

### Architecture

```
User Query
    │
    ▼
┌─────────────────────┐
│   LangChain Agent   │
│   (Orchestrator)    │
└────────┬────────────┘
         │
    ┌────┴────┐
    │         │
    ▼         ▼
┌────────┐ ┌──────────────────┐
│ Oracle │ │  Spraay x402     │
│ AI DB  │ │  Gateway         │
│ (RAG)  │ │  (Paid Inference)│
└────────┘ └──────────────────┘
    │              │
    ▼              ▼
 Context    Premium Response
    │              │
    └──────┬───────┘
           ▼
     Final Answer
```

### Prerequisites

- Python 3.9+
- Oracle Database 23ai (or Oracle Autonomous Database with AI Vector Search)
- An Ethereum wallet with USDC on Base (for x402 payments)
- OpenAI API key (for embeddings)


## 1. Environment Setup

Install the required packages. We use `langchain-oracledb` for Oracle AI Database integration, `httpx` for HTTP calls to the Spraay gateway, and standard crypto libraries for x402 payment signing.


In [ ]:
# Install dependencies
%pip install --quiet \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-oracledb \
    oracledb \
    httpx \
    eth-account \
    python-dotenv

## 2. Configuration

Set up your environment variables. You'll need:
- Oracle Database connection details
- OpenAI API key (for embeddings)
- An Ethereum private key with USDC on Base (for x402 payments)

> ⚠️ **Security**: Never commit private keys to version control. Use environment variables or a `.env` file.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Oracle Database Configuration
ORACLE_USER = os.getenv("ORACLE_USER", "your_user")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD", "your_password")
ORACLE_DSN = os.getenv("ORACLE_DSN", "your_host:1521/your_service")

# OpenAI (for embeddings)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Spraay x402 Gateway
SPRAAY_GATEWAY_URL = "https://gateway.spraay.app"

# Ethereum wallet for x402 payments (must hold USDC on Base)
WALLET_PRIVATE_KEY = os.getenv("WALLET_PRIVATE_KEY")

print("✅ Configuration loaded")

## 3. Connect to Oracle AI Database

Establish a connection to Oracle Database and set up the vector store for RAG. Oracle AI Database supports native vector search with `AI Vector Search`, making it ideal for agent memory and context retrieval.


In [ ]:
import oracledb

# Connect to Oracle Database
connection = oracledb.connect(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN
)

print(f"✅ Connected to Oracle Database: {connection.version}")

## 4. Set Up Oracle Vector Store

We'll create a vector table and populate it with sample documents. In production, this would contain your knowledge base — product docs, internal wikis, compliance data, etc.

The Oracle vector store uses `langchain-oracledb` for native integration with LangChain's retrieval chain.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_oracledb import OracleVS
from langchain_core.documents import Document

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY
)

# Sample knowledge base documents
# In production, load from your actual data sources
documents = [
    Document(
        page_content="Spraay Protocol is a multi-chain batch payment protocol deployed across 15 blockchains "
                     "including Base, Ethereum, Bitcoin, Arbitrum, Polygon, BNB Chain, Avalanche, Unichain, "
                     "Plasma, BOB, Bittensor, Solana, Stacks, Stellar, and XRP Ledger. It enables sending "
                     "payments to 200+ recipients in a single transaction, reducing gas costs by up to 80%.",
        metadata={"source": "spraay_docs", "topic": "overview"}
    ),
    Document(
        page_content="The x402 payment protocol uses HTTP status code 402 (Payment Required) to enable "
                     "machine-to-machine payments. When a server returns a 402 response, the client's payment "
                     "middleware automatically handles USDC payment on Base mainnet and retries the request "
                     "with a payment proof in the X-PAYMENT header.",
        metadata={"source": "x402_spec", "topic": "protocol"}
    ),
    Document(
        page_content="Oracle AI Vector Search enables semantic similarity search directly in the database. "
                     "Vectors can be stored alongside relational data, enabling hybrid queries that combine "
                     "traditional SQL filtering with vector similarity search in a single query.",
        metadata={"source": "oracle_docs", "topic": "vector_search"}
    ),
    Document(
        page_content="AI agent payments solve the API key management problem. Instead of provisioning and "
                     "rotating API keys for each service, agents pay per-request with USDC micropayments. "
                     "This is more secure (no stored credentials) and more flexible (pay only for what you use).",
        metadata={"source": "agent_payments", "topic": "benefits"}
    ),
    Document(
        page_content="The Spraay x402 Gateway provides 88 paid endpoints across 19 categories including "
                     "AI inference (200+ models), Bittensor decentralized AI (43+ models via SN64 Chutes AI), "
                     "search/RAG (Tavily-powered), batch payments, escrow, swaps, bridges, RPC (Alchemy), "
                     "IPFS storage (Pinata), robotics (RTP), supply chain (SCTP), and ERC-4337 agent wallets. "
                     "Each endpoint costs fractions of a cent in USDC per request.",
        metadata={"source": "spraay_docs", "topic": "gateway_endpoints"}
    ),
    Document(
        page_content="Spraay's batch payment tool is an official community tool in Google's Agent Development "
                     "Kit (ADK), merged as PR #95 in google/adk-python-community. This means any Google ADK "
                     "agent can natively batch-send crypto payments via Spraay's smart contract on Base.",
        metadata={"source": "spraay_docs", "topic": "google_adk"}
    ),
]

# Create Oracle Vector Store
vector_store = OracleVS.from_documents(
    documents,
    embeddings,
    client=connection,
    table_name="SPRAAY_AGENT_KNOWLEDGE",
    distance_strategy="COSINE"
)

print(f"✅ Oracle Vector Store created with {len(documents)} documents")

## 5. Build the x402 Payment Client

The x402 payment flow works like this:

1. Agent makes an HTTP request to a paid endpoint
2. Server responds with `402 Payment Required` + payment requirements (price, recipient, token)
3. Client signs a USDC payment authorization on Base
4. Client retries the request with `X-PAYMENT` header containing the signed payment
5. Server verifies payment and returns the response

We'll build a lightweight client that handles this flow automatically.


In [ ]:
import httpx
import json
import time
from eth_account import Account
from eth_account.messages import encode_defunct

class SpraayX402Client:
    """Client for making x402 payments to the Spraay Gateway.
    
    Gateway: https://gateway.spraay.app
    Docs: https://docs.spraay.app
    88 endpoints across 19 categories, all payable with USDC on Base.
    """

    def __init__(self, gateway_url: str, private_key: str):
        self.gateway_url = gateway_url.rstrip("/")
        self.account = Account.from_key(private_key)
        self.client = httpx.Client(timeout=30.0)
        self.payment_log = []

    def _sign_payment(self, payment_requirements: dict) -> str:
        """Sign a USDC payment authorization for x402."""
        # Extract payment details from 402 response
        amount = payment_requirements.get("maxAmountRequired", "0")
        recipient = payment_requirements.get("resource")
        network = payment_requirements.get("network", "base")

        # Create payment payload
        payment = {
            "x402Version": 1,
            "scheme": "exact",
            "network": network,
            "payload": {
                "signature": "",
                "authorization": {
                    "from": self.account.address,
                    "to": recipient,
                    "value": amount,
                    "validAfter": 0,
                    "validBefore": int(time.time()) + 3600,
                    "nonce": hex(int(time.time() * 1000)),
                }
            }
        }

        # Sign the authorization
        message = json.dumps(payment["payload"]["authorization"], sort_keys=True)
        signed = self.account.sign_message(encode_defunct(text=message))
        payment["payload"]["signature"] = signed.signature.hex()

        return json.dumps(payment)

    def request(self, endpoint: str, method: str = "GET", **kwargs) -> dict:
        """Make an x402-enabled request to the Spraay Gateway."""
        url = f"{self.gateway_url}{endpoint}"

        # First attempt — may return 402
        response = self.client.request(method, url, **kwargs)

        if response.status_code == 402:
            # Parse payment requirements
            requirements = response.json()
            print(f"  💰 Payment required: {requirements.get('maxAmountRequired', 'unknown')} USDC")

            # Sign payment
            payment_header = self._sign_payment(requirements)

            # Retry with payment
            headers = kwargs.get("headers", {})
            headers["X-PAYMENT"] = payment_header
            kwargs["headers"] = headers

            response = self.client.request(method, url, **kwargs)

            # Log the payment
            self.payment_log.append({
                "endpoint": endpoint,
                "amount": requirements.get("maxAmountRequired"),
                "timestamp": time.time()
            })

        if response.status_code == 200:
            return response.json()
        else:
            return {"error": f"Request failed with status {response.status_code}"}

    def get_payment_summary(self) -> dict:
        """Get a summary of all payments made in this session."""
        total = sum(float(p.get("amount", 0)) for p in self.payment_log)
        return {
            "total_payments": len(self.payment_log),
            "total_usdc_spent": total,
            "payments": self.payment_log
        }


# Initialize the x402 client
x402_client = SpraayX402Client(
    gateway_url=SPRAAY_GATEWAY_URL,
    private_key=WALLET_PRIVATE_KEY
)

print(f"✅ x402 Client initialized")
print(f"   Wallet: {x402_client.account.address}")
print(f"   Gateway: {SPRAAY_GATEWAY_URL}")

## 6. Define Agent Tools

Now we create LangChain tools that the agent can use:

1. **`search_knowledge_base`** — Retrieves relevant context from Oracle AI Database (free)
2. **`spraay_ai_inference`** — Calls 200+ AI models via Spraay's x402 gateway ($0.04 USDC/request)
3. **`spraay_web_search`** — Searches the web via Tavily through Spraay ($0.02 USDC/request)

The agent decides which tools to use based on the query. Free retrieval first, paid inference only when needed.


In [ ]:
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str) -> str:
    """Search the Oracle AI Database vector store for relevant context.
    Use this FIRST to check if the answer exists in our knowledge base.
    This is a free operation — no payment required."""

    results = vector_store.similarity_search(query, k=3)

    if not results:
        return "No relevant documents found in the knowledge base."

    context = "\n\n".join([
        f"[Source: {doc.metadata.get('source', 'unknown')}] {doc.page_content}"
        for doc in results
    ])
    return f"Found {len(results)} relevant documents:\n\n{context}"


@tool
def spraay_ai_inference(prompt: str) -> str:
    """Call a premium AI model via the Spraay x402 Gateway.
    Costs $0.04 USDC per request. Uses /api/v1/chat/completions (200+ models).
    Use ONLY when the knowledge base doesn't have the answer."""

    print(f"  🤖 Calling AI inference via Spraay x402 ($0.04 USDC)...")
    response = x402_client.request(
        "/api/v1/chat/completions",
        method="POST",
        json={
            "model": "gpt-4",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 500
        }
    )

    if "error" in response:
        return f"Inference failed: {response['error']}"

    return response.get("choices", [{}])[0].get("message", {}).get("content", "No response")


@tool
def spraay_bittensor_inference(prompt: str) -> str:
    """Call decentralized AI via Bittensor through Spraay's x402 Gateway.
    Costs $0.03 USDC per request. Uses /bittensor/v1/chat/completions.
    OpenAI-compatible drop-in routed through Bittensor SN64 (Chutes AI).
    43+ models including DeepSeek, Qwen, Llama, Mistral."""

    print(f"  🧬 Calling Bittensor inference via Spraay x402 ($0.03 USDC)...")
    response = x402_client.request(
        "/bittensor/v1/chat/completions",
        method="POST",
        json={
            "model": "deepseek-ai/DeepSeek-V3-0324",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 500
        }
    )

    if "error" in response:
        return f"Bittensor inference failed: {response['error']}"

    return response.get("choices", [{}])[0].get("message", {}).get("content", "No response")


@tool
def spraay_web_search(query: str) -> str:
    """Search the web via Spraay's x402 search endpoint (Tavily-powered).
    Costs $0.02 USDC per request. Uses /api/v1/search/web.
    Use when you need current information not available in the knowledge base."""

    print(f"  🔍 Searching the web via Spraay x402 ($0.02 USDC)...")
    response = x402_client.request(
        "/api/v1/search/web",
        method="POST",
        json={"query": query, "search_depth": "basic", "max_results": 5}
    )

    if "error" in response:
        return f"Search failed: {response['error']}"

    results = response.get("results", [])
    if not results:
        return "No web results found."

    formatted = "\n".join([
        f"- {r.get('title', 'Untitled')}: {r.get('snippet', 'No snippet')}"
        for r in results[:5]
    ])
    return f"Web search results:\n{formatted}"


tools = [search_knowledge_base, spraay_ai_inference, spraay_bittensor_inference, spraay_web_search]
print(f"✅ {len(tools)} agent tools registered")
print("   - search_knowledge_base      (free — Oracle AI DB)")
print("   - spraay_ai_inference         ($0.04 USDC — 200+ models)")
print("   - spraay_bittensor_inference  ($0.03 USDC — Bittensor SN64)")
print("   - spraay_web_search           ($0.02 USDC — Tavily)")

## 7. Build the Payment-Enabled Agent

We create a LangChain ReAct agent that:
1. **Always checks the knowledge base first** (free)
2. **Escalates to paid inference** only when local context is insufficient
3. **Can choose between centralized AI or decentralized Bittensor inference**
4. **Logs all payments** for transparency and auditing

This cost-aware behavior is guided by the system prompt — the agent is instructed to prefer free tools and only pay when necessary.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import ChatPromptTemplate

# Initialize LLM for agent reasoning
llm = ChatOpenAI(
    model="gpt-4",
    temperature=0,
    openai_api_key=OPENAI_API_KEY
)

# System prompt that enforces cost-aware behavior
system_prompt = """You are an AI research agent with access to both free and paid tools.

IMPORTANT RULES:
1. ALWAYS search the knowledge base first using search_knowledge_base (this is FREE)
2. Only use paid tools if the knowledge base doesn't have a sufficient answer
3. For AI analysis, prefer spraay_ai_inference ($0.04) for complex tasks or
   spraay_bittensor_inference ($0.03) for decentralized/censorship-resistant inference
4. Use spraay_web_search ($0.02) only for current/real-time information
5. When using paid tools, be specific in your queries to minimize cost
6. After answering, briefly note which tools you used and whether any payments were made

Available tools:
{tools}

Tool names: {tool_names}

Use this format:
Question: the question to answer
Thought: think about which tool to use (prefer free tools first)
Action: the tool to use
Action Input: the input to the tool
Observation: the result
... (repeat as needed)
Thought: I now know the final answer
Final Answer: the answer with sources noted

Question: {input}
{agent_scratchpad}"""

prompt = ChatPromptTemplate.from_template(system_prompt)

# Create the agent
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

print("✅ Payment-enabled agent ready")

## 8. Run the Agent

Let's test the agent with different types of queries:
- **Query 1**: Answerable from the knowledge base (should be free)
- **Query 2**: Requires premium inference (should trigger x402 payment)
- **Query 3**: Requires current web data (should trigger x402 payment)


In [ ]:
# Query 1: Should be answered from Oracle AI DB (free)
print("=" * 60)
print("QUERY 1: Knowledge base lookup (expected: FREE)")
print("=" * 60)

result1 = agent_executor.invoke({
    "input": "What is the x402 payment protocol and how does it work?"
})

print(f"\n📝 Answer: {result1['output']}")

In [ ]:
# Query 2: Requires premium AI inference (expected: PAID)
print("=" * 60)
print("QUERY 2: Analysis requiring premium inference (expected: PAID)")
print("=" * 60)

result2 = agent_executor.invoke({
    "input": "Compare the economic efficiency of x402 micropayments vs traditional "
             "API key subscriptions for AI agent deployments at scale. "
             "Include specific cost breakdowns."
})

print(f"\n📝 Answer: {result2['output']}")

In [ ]:
# Query 3: Current web data (expected: PAID)
print("=" * 60)
print("QUERY 3: Current information lookup (expected: PAID)")
print("=" * 60)

result3 = agent_executor.invoke({
    "input": "What are the latest developments in the x402 payment protocol ecosystem?"
})

print(f"\n📝 Answer: {result3['output']}")

## 9. Payment Audit Trail

One of the key benefits of x402 is full payment transparency. Every micropayment is an on-chain USDC transfer on Base, providing a complete audit trail. Let's review what the agent spent during this session.


In [ ]:
# Review all payments made during this session
summary = x402_client.get_payment_summary()

print("=" * 60)
print("💰 PAYMENT SUMMARY")
print("=" * 60)
print(f"Total requests with payment: {summary['total_payments']}")
print(f"Total USDC spent:            ${summary['total_usdc_spent']:.6f}")
print()

if summary['payments']:
    print("Individual payments:")
    for i, payment in enumerate(summary['payments'], 1):
        print(f"  {i}. {payment['endpoint']} — ${float(payment.get('amount', 0)):.6f} USDC")
else:
    print("No paid requests were made (all answered from knowledge base)")

print()
print("All payments are verifiable on-chain at:")
print("https://basescan.org/address/" + x402_client.account.address)

## 10. Bonus: Oracle Hybrid Search with Payment Context

Oracle AI Database supports hybrid search — combining vector similarity with traditional SQL filtering in a single query. This is powerful for agents that need to search payment history alongside semantic context.


In [ ]:
# Example: Hybrid search combining vector similarity with metadata filtering
# This shows Oracle's unique ability to mix SQL + vector search

hybrid_results = vector_store.similarity_search(
    "payment protocol benefits",
    k=3,
    filter={"topic": "benefits"}  # SQL filter on metadata
)

print("🔎 Hybrid Search Results (vector + SQL filter):")
print(f"   Filter: topic = 'benefits'")
print(f"   Found: {len(hybrid_results)} documents")
print()
for doc in hybrid_results:
    print(f"   [{doc.metadata.get('source')}] {doc.page_content[:120]}...")

## 11. Cleanup


In [ ]:
# Close connections
x402_client.client.close()
connection.close()
print("✅ Connections closed")

## Summary

This notebook demonstrated how to build a **payment-enabled AI agent** that combines:

| Component | Role | Cost |
|-----------|------|------|
| **Oracle AI Database** | Vector storage, semantic search, hybrid queries | Free (self-hosted) |
| **Spraay x402 Gateway** | 88 endpoints: AI inference, Bittensor, search, batch payments, and more | Pay-per-request (USDC on Base) |
| **LangChain** | Agent orchestration, tool routing, reasoning | Free (open source) |

### Key Takeaways

1. **Cost-aware agents** — The agent checks free resources (Oracle AI DB) before escalating to paid services (Spraay x402), minimizing spend automatically.

2. **No API keys needed** — x402 replaces API key management with per-request USDC micropayments. No credentials to rotate, no subscriptions to manage.

3. **Centralized + Decentralized AI** — Agents can choose between 200+ centralized models (`/api/v1/chat/completions` at $0.04) or 43+ decentralized Bittensor models (`/bittensor/v1/chat/completions` at $0.03) — same OpenAI-compatible interface.

4. **Full audit trail** — Every payment is an on-chain USDC transfer on Base, providing transparent, verifiable spend tracking.

5. **Oracle's hybrid search** — Combining vector similarity with SQL filtering in a single query is uniquely powerful for agents that need structured + semantic retrieval.

6. **Production-ready integrations** — Spraay's batch payment tool is merged into Google's ADK (PR #95 in `google/adk-python-community`), with integrations across LangChain, CrewAI, and more.

### Gateway at a Glance

| Stat | Value |
|------|-------|
| Endpoints | 88 |
| Categories | 19 |
| Chains | 15 |
| AI Models | 200+ (centralized) + 43+ (Bittensor) |
| Payment | USDC on Base |
| Contract | [`0x1646452F98E36A3c9Cfc3eDD8868221E207B5eEC`](https://basescan.org/address/0x1646452F98E36A3c9Cfc3eDD8868221E207B5eEC) |

### Learn More

- **Spraay Protocol**: [spraay.app](https://spraay.app)
- **Spraay x402 Gateway Docs**: [docs.spraay.app](https://docs.spraay.app)
- **Gateway**: [gateway.spraay.app](https://gateway.spraay.app)
- **MCP Server**: [Smithery — @plagtech/spraay-x402-mcp](https://smithery.ai/server/@plagtech/spraay-x402-mcp) (60 tools for Claude Desktop, Cursor, Cline)
- **Google ADK Tool**: [google/adk-python-community](https://github.com/google/adk-python-community) (PR #95)
- **x402 Protocol Spec**: [x402.org](https://www.x402.org)
- **Oracle AI Vector Search**: [oracle.com/database/ai](https://www.oracle.com/database/ai/)
- **LangChain Oracle Integration**: [langchain-oracledb](https://python.langchain.com/docs/integrations/vectorstores/oracle/)
- **GitHub**: [github.com/plagtech](https://github.com/plagtech)
- **Twitter**: [@Spraay_app](https://twitter.com/Spraay_app)

---

*Contributed by [Spraay Protocol](https://spraay.app) — Multi-chain batch payments & x402 payment gateway for AI agents.*
